In [1]:
import vrep 
import sys
import time 
import numpy as np
from tank import *
import skfuzzy
from skfuzzy import control as ctrl

In [ ]:
def find_place(dist_NW, dist_WN, dist_NE):
    distance_WN = ctrl.Antecedent(np.arange(0, 5, 0.01), 'distance_WN')
    distance_NW = ctrl.Antecedent(np.arange(0, 6, 0.01), 'distance_NW')
    distance_NE = ctrl.Antecedent(np.arange(0, 6, 0.01), 'distance_NE')

    speed = ctrl.Consequent(np.arange(-1, 7, 0.01), 'speed')

    distance_WN['close'] = skfuzzy.trapmf(distance_WN.universe, [0, 0, 1.5, 3])
    distance_WN['far'] = skfuzzy.trapmf(distance_WN.universe, [1.5, 2.51, 5, 5])

    distance_NW['close'] = skfuzzy.trapmf(distance_NW.universe, [0, 0, 1.5, 2])
    distance_NW['far'] = skfuzzy.trapmf(distance_NW.universe, [1, 2, 4, 5])

    distance_NE['close'] = skfuzzy.trapmf(distance_NE.universe, [0, 0, 1.5, 2])
    distance_NE['far'] = skfuzzy.trapmf(distance_NE.universe, [1, 2, 4, 5])

    speed['stop'] = skfuzzy.trimf(speed.universe, [-1, 0, 1])
    speed['break'] = skfuzzy.trimf(speed.universe, [0, 2, 4])
    speed['go'] = skfuzzy.trimf(speed.universe, [3, 5, 7])

    rules = (
        ctrl.Rule(distance_WN['close'] & distance_NW['close'] & distance_NE['close'], speed['go']),
        ctrl.Rule(distance_WN['far'] & distance_WN['close'] & distance_NE['far'], speed['stop']),
        ctrl.Rule(distance_WN['close'] & distance_NW['far'], speed['go']),
        ctrl.Rule(distance_WN['far'] & distance_NW['far'], speed['go'])
    )

    sim_ctrl = ctrl.ControlSystem(rules)
    sim = ctrl.ControlSystemSimulation(sim_ctrl)

    sim.input['distance_NW'] = dist_NW
    sim.input['distance_WN'] = dist_WN
    sim.input['distance_NE'] = dist_NE
    sim.compute()

    return sim.output['speed']

def park(dist_WN, dist_EN, dist_WS, dist_ES):
    distance_WN = ctrl.Antecedent(np.arange(0, 5.01, 0.01), 'distance_WN')
    distance_EN = ctrl.Antecedent(np.arange(0, 6.01, 0.01), 'distance_EN')
    distance_WS = ctrl.Antecedent(np.arange(0, 6.01, 0.01), 'distance_WS')
    distance_ES = ctrl.Antecedent(np.arange(0, 6.01, 0.01), 'distance_ES')

    speed = ctrl.Consequent(np.arange(-1, 5, 0.01), 'speed')

    distance_WN['close'] = skfuzzy.trapmf(distance_WN.universe, [0, 0, 1, 2])
    distance_WN['far'] = skfuzzy.trapmf(distance_WN.universe, [1.5, 2.51, 5, 5])

    distance_EN['close'] = skfuzzy.trapmf(distance_EN.universe, [0, 0, 1.5, 2])
    distance_EN['far'] = skfuzzy.trapmf(distance_EN.universe, [1, 2, 3, 6])

    distance_WS['close'] = skfuzzy.trapmf(distance_WS.universe, [0, 0, 1.5, 2])
    distance_WS['far'] = skfuzzy.trapmf(distance_WS.universe, [1, 2, 3, 6])

    distance_ES['close'] = skfuzzy.trapmf(distance_ES.universe, [0, 0, 1.5, 2])
    distance_ES['far'] = skfuzzy.trapmf(distance_ES.universe, [1, 2, 3, 6])

    speed['stop'] = skfuzzy.trimf(speed.universe, [-1, 0, 1])
    speed['break'] = skfuzzy.trimf(speed.universe, [0, 1, 2])
    speed['go'] = skfuzzy.trimf(speed.universe, [1, 2, 5])

    rules = (
        ctrl.Rule(distance_WN['far'] & distance_EN['far'], speed['go']),
        ctrl.Rule(distance_WN['far'] & distance_EN['far'], speed['break']),
        ctrl.Rule(distance_WN['close'] & distance_EN['close'], speed['break']),
        ctrl.Rule(distance_WN['close'] & distance_EN['far'], speed['break']),

        ctrl.Rule(distance_WS['far'] & distance_ES['close'], speed['go']),
        ctrl.Rule(distance_WS['far'] & distance_ES['far'], speed['break']),
        ctrl.Rule(distance_WS['close'] & distance_ES['close'], speed['stop']),
        ctrl.Rule(distance_WS['close'] & distance_ES['far'], speed['break']),
    )

    sim_ctrl = ctrl.ControlSystem(rules)
    sim = ctrl.ControlSystemSimulation(sim_ctrl)

    sim.input['distance_WN'] = dist_WN
    sim.input['distance_EN'] = dist_EN
    sim.input['distance_ES'] = dist_ES
    sim.input['distance_WS'] = dist_WS

    sim.compute()

    return sim.output['speed']

In [115]:
vrep.simxFinish(-1) # closes all opened connections, in case any prevoius wasnt finished
clientID=vrep.simxStart('127.0.0.1',19999,True,True,5000,5) # start a connection

if clientID!=-1:
    print ("Connected to remote API server")
else:
    print("Not connected to remote API server")
    sys.exit("Could not connect")

#create instance of Tank
tank=Tank(clientID)

Connected to remote API server


In [116]:
proximity_sensors=["EN","ES","NE","NW","SE","SW","WN","WS"]
proximity_sensors_handles=[0]*8

# get handle to proximity sensors
for i in range(len(proximity_sensors)):
    err_code,proximity_sensors_handles[i] = vrep.simxGetObjectHandle(clientID,"Proximity_sensor_"+proximity_sensors[i], vrep.simx_opmode_blocking)
    
#read and print values from proximity sensors
#first reading should be done with simx_opmode_streaming, further with simx_opmode_buffer parameter
for sensor_name, sensor_handle in zip(proximity_sensors,proximity_sensors_handles):
        err_code,detectionState,detectedPoint,detectedObjectHandle,detectedSurfaceNormalVector=vrep.simxReadProximitySensor(clientID,sensor_handle,vrep.simx_opmode_streaming)

In [117]:
tank.forward(5)

distances = dict()
detection_states = dict()

for sensor_name, sensor_handle in zip(proximity_sensors,proximity_sensors_handles):
    err_code,detectionState,detectedPoint,detectedObjectHandle,detectedSurfaceNormalVector=vrep.simxReadProximitySensor(clientID,sensor_handle,vrep.simx_opmode_buffer)
    distances[sensor_name] = np.linalg.norm(detectedPoint)
    detection_states[sensor_name] = detectionState

stages = [
     'find_place',
     'park',
     'go_forward'
]
counter = 0
stage = 'find_place'
#continue reading and printing values from proximity sensors
t = time.time()
while (time.time()-t)<100: # read values for 5 seconds
    counter += 1
    for sensor_name, sensor_handle in zip(proximity_sensors,proximity_sensors_handles):
        err_code,detectionState,detectedPoint,detectedObjectHandle,detectedSurfaceNormalVector=vrep.simxReadProximitySensor(clientID,sensor_handle,vrep.simx_opmode_buffer )
        if(err_code == 0):
            # print("Proximity_sensor_"+sensor_name, np.linalg.norm(detectedPoint))
            distances[sensor_name] = np.linalg.norm(detectedPoint)
            detection_states[sensor_name] = detectionState
    
    if stage == 'find_place':
        tank_speed = find_place(distances['NW'], distances['WN'], distances['NE'])
        if tank_speed < 3.45:
            tank.stop()
            stage = 'park'
            print("Zmiana na fazę parkowania")
        else: 
            tank.forward(tank_speed)
    if stage == 'park':
        tank_speed = park(distances['WN'], distances['EN'], distances['WS'], distances['ES'])
        if counter % 2000:
            # print(distances['WN'], distances['EN'], distances['WS'], distances['ES'])
            print(tank_speed)
        if tank_speed < 0.51:
            tank.forward(1)
            stage = 'go_forward'
            print("Faza wyrównania")
        else:
            tank.leftvelocity = tank_speed
            tank.rightvelocity = tank_speed * 2
            tank.setVelocity()
            # print(tank.leftvelocity, tank.rightvelocity)
            tank.go()
    if stage == 'go_forward':
        tank.forward(3)
        if not detection_states['NW'] and not detection_states['NE']:
            tank.stop()
            break


    # print()

Zmiana na fazę parkowania
2.587943642140883
2.587943642140883
2.587943642140883
2.587943642140883
2.587943642140883
2.587254316810007
2.587254316810007
2.587254316810007
2.587254316810007
2.587904660563012
2.587904660563012
2.587904660563012
2.587904660563012
2.587904660563012
2.5861351564531954
2.5861351564531954
2.5861351564531954
2.5861351564531954
2.631860070514438
2.631860070514438
2.631860070514438
2.631860070514438
2.631860070514438
2.631860070514438
2.6018380781122943
2.6018380781122943
2.6018380781122943
2.6018380781122943
2.602062260842443
2.602062260842443
2.602062260842443
2.602062260842443
2.296238898365274
2.296238898365274
2.296238898365274
2.296238898365274
2.296238898365274
2.2913478565208476
2.2913478565208476
2.2913478565208476
2.2913478565208476
2.2913478565208476
2.2889842290349978
2.2889842290349978
2.2889842290349978
2.2889842290349978
2.2854953744144217
2.2854953744144217
2.2854953744144217
2.2854953744144217
2.2854953744144217
2.281403741882968
2.28140374188296